In [2]:
from utils import *
import numpy as np
import torch
from torch.nn.functional import silu

In [3]:
path = ROOT.parent / 'data' / 'invert' / 'from_val_rk4'

In [4]:
class PixelsDataset():
    def __init__(self, path, split):
        Z, _, _ = load_invert(path)
        N, L, C, H, W = Z.shape
        N = int(N*0.8)
        if split == 'train':
            Z = Z[:N ,...]
        elif split == 'val':
            Z = Z[N: ,...]

        self.size = Z.shape[0] * (L - 1) * C * H * W
        self.H = H
        self.W = W
        self.C = C
        self.L = L

        self.Z_t = torch.from_numpy(Z)
        self.xs = torch.arange(H, dtype=torch.float32)
        self.ys = torch.arange(W, dtype=torch.float32)
        self.leads = torch.arange(2, L+1, dtype=torch.float32)  # leads 2,...,24 in hours

        batch_numbers = (L-1) * 15
        self.batch_size = self.size // batch_numbers

    def iterate_batches(self):
        # int32 and no .tolist(): keeps an exact shuffle at 4 bytes per index
        perm = torch.randperm(self.size, dtype=torch.int32)
        for start in range(0, self.size, self.batch_size):
            idx = perm[start:start + self.batch_size]
            y = idx % self.W
            idx = idx // self.W
            x = idx % self.H
            idx = idx // self.H
            ch = idx % self.C
            idx = idx // self.C
            t = idx % (self.L - 1)
            n = idx // (self.L - 1)
            # Z index t+1 is lead t+2, which is what leads[t] encodes
            yield self.leads[t], ch, self.xs[x], self.ys[y], self.Z_t[n, t, ch, x, y], self.Z_t[n, t + 1, ch, x, y]

train_ds = PixelsDataset(path, split='train')
# test code
lead, ch, x, y, z_prev, z_next = next(train_ds.iterate_batches())
print(int(train_ds.size/(0.8*23*2*64*64))) 
print(lead.shape, ch.shape, x.shape, y.shape, z_prev.shape, z_next.shape)
print(f"x max: {torch.max(x)}, x min {torch.min(x)}")
print(f"y max: {torch.max(y)}, y min {torch.min(y)}")
print(f"lead max: {torch.max(lead)}, lead min {torch.min(lead)}")
        

149
torch.Size([65536]) torch.Size([65536]) torch.Size([65536]) torch.Size([65536]) torch.Size([65536]) torch.Size([65536])
x max: 63.0, x min 0.0
y max: 63.0, y min 0.0
lead max: 24.0, lead min 2.0


In [5]:
val_ds = PixelsDataset(path, split='val')
print(int(val_ds.size/(0.2*23*2*64*64))) 
print(val_ds.batch_size)

149
16384


In [13]:
class PositionalEmbedding(torch.nn.Module):
    def __init__(self, num_channels, max_positions=24, endpoint=False):
        super().__init__()
        self.num_channels = num_channels
        self.max_positions = max_positions
        self.endpoint = endpoint

    def forward(self, x):
        #FROMTRACE:
        # num_channels decide how long the output will be
        # max_position decides the slowest wave
        
        # Define the frequencies domain from [0, num_channels//2]
        # Normalise the frequencies domain to [0, 1]
        # For each f: x = x * max_positions^(-f)
        freqs = torch.arange(start=0, end=self.num_channels //
                             2, dtype=torch.float32, device=x.device)
        freqs = freqs / (self.num_channels // 2 - (1 if self.endpoint else 0))
        freqs = (1 / self.max_positions) ** freqs
        x = x.ger(freqs.to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

class FourierEmbedding(torch.nn.Module):
    def __init__(self, num_channels, period=64):
        super().__init__()
        n_freq = num_channels // 2                    # F = num_channels / 2
        freqs = torch.arange(1, n_freq + 1, dtype=torch.float32) / period
        self.register_buffer('freqs', freqs)   

    def forward(self, x):
        # Outer product
        x = x.ger((2 * np.pi * self.freqs).to(x.dtype))
        x = torch.cat([x.cos(), x.sin()], dim=1)
        return x

class TimeVaryingOU(torch.nn.Module):
    # embedding t with positional embedding
    # embedding x with fourier embedding
    # embedding y with fourier embedding
    # embedding ch with nn.Embedding(2, d)
    # 3-4 MLP layers with SiLU
    # end up with softplus 
    def __init__(self, emb_dim=64, hidden_dim=256, n_hidden=3, xy_dim=8):
        super().__init__()
        self.t_emb = PositionalEmbedding(emb_dim)
        self.x_emb = FourierEmbedding(xy_dim)
        self.y_emb = FourierEmbedding(xy_dim)
        self.ch_emb = torch.nn.Embedding(2, emb_dim)

        in_dim = emb_dim + 2 * xy_dim + emb_dim 
        dims = [in_dim] + [hidden_dim] * n_hidden
        self.layers = torch.nn.ModuleList(
            torch.nn.Linear(d_in, d_out) for d_in, d_out in zip(dims[:-1], dims[1:])
        )
        self.out = torch.nn.Linear(hidden_dim, 2)
        torch.nn.init.zeros_(self.out.weight)
        self.out.bias.data.copy_(torch.tensor([0.541, 0.07]))   # sigma = 1, alpha ≈ 0.7

    def forward(self, t, x, y, ch):
        embedded_x = self.x_emb(x)
        embedded_y = self.y_emb(y)
        embedded_ch = self.ch_emb(ch)
        embedded_t = self.t_emb(t)
        h = torch.cat([embedded_t, embedded_x, embedded_y, embedded_ch], dim=1)
        for layer in self.layers:
            h = silu(layer(h))
        h = self.out(h)
        sigma = torch.nn.functional.softplus(h[:, 0]) + 1e-6
        c2 = torch.nn.functional.softplus(h[:, 1]) + 1e-6
        return sigma, c2


In [7]:
def batch_loss(batch):
    lead, ch, x, y, z_prev, z_next = (b.to(device) for b in batch)
    sigma_next, c2 = model(lead, x, y, ch)          # sigma at lead k, c^2 on interval [k-1, k]
    sigma_prev, _ = model(lead - 1, x, y, ch)       # sigma at lead k-1
    alpha = torch.exp(-0.5 * c2)                    # dt = 1 h
    one_m_a2 = -torch.expm1(-c2)                    # 1 - alpha^2, stable as alpha -> 1
    xi_prev, xi_next = z_prev / sigma_prev, z_next / sigma_next
    loss = 0.5 * (torch.log(sigma_next**2) + torch.log(one_m_a2) + (xi_next - alpha * xi_prev)**2 / one_m_a2)
    first = lead == 2                               # transition 1 -> 2 also carries the lead-1 marginal
    loss = loss + first * 0.5 * (torch.log(sigma_prev**2) + xi_prev**2)
    return loss.mean(), len(lead)

In [15]:
def train(model, train_ds, val_ds, device, epochs=30, lr=1e-4):
    # dt is the gap between consecutive leads in hours, so c is in physical units
    model = model.to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)

    history = {'train': [], 'val': []}
    for epoch in range(epochs):
        model.train()
        total = 0.0
        for batch in train_ds.iterate_batches():
            loss, n = batch_loss(batch)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * n
        history['train'].append(total / train_ds.size)

        model.eval()
        total = 0.0
        with torch.no_grad():
            for batch in val_ds.iterate_batches():
                loss, n = batch_loss(batch)
                total += loss.item() * n
        history['val'].append(total / val_ds.size)

        print(f"epoch {epoch + 1}/{epochs}  train nll {history['train'][-1]:.4f}  val nll {history['val'][-1]:.4f}")
    return history


model = TimeVaryingOU()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
history = train(model, train_ds, val_ds, device)

epoch 1/30  train nll 0.1106  val nll 0.1161
epoch 2/30  train nll 0.1096  val nll 0.1161
epoch 3/30  train nll 0.1095  val nll 0.1161
epoch 4/30  train nll 0.1095  val nll 0.1163
epoch 5/30  train nll 0.1095  val nll 0.1163
epoch 6/30  train nll 0.1095  val nll 0.1162
epoch 7/30  train nll 0.1094  val nll 0.1162
epoch 8/30  train nll 0.1094  val nll 0.1162
epoch 9/30  train nll 0.1094  val nll 0.1161
epoch 10/30  train nll 0.1093  val nll 0.1163
epoch 11/30  train nll 0.1093  val nll 0.1161
epoch 12/30  train nll 0.1093  val nll 0.1162
epoch 13/30  train nll 0.1092  val nll 0.1162
epoch 14/30  train nll 0.1092  val nll 0.1162
epoch 15/30  train nll 0.1092  val nll 0.1162


KeyboardInterrupt: 